# ECG New Image Testing Notebook

This notebook validates the ECG preprocessing pipeline using a manually uploaded ECG image.

Pipeline Steps:
1. Upload ECG image
2. Apply preprocessing pipeline
3. Extract 12-lead normalized signals
4. Flatten to fixed-length feature vector
5. Save structured CSV output

This notebook is for validation purposes only.
Production logic resides in `ECG_Preprocessing_Functions.py`.

# Import Module

In [12]:
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import tempfile

from ECG_Preprocessing_Functions import process_single_ecg_image, TARGET_LEAD_LENGTH

# Upload Widget

In [13]:
uploader = widgets.FileUpload(
    accept=".jpg,.png,.jpeg",
    multiple=False
)

display(uploader)

FileUpload(value=(), accept='.jpg,.png,.jpeg', description='Upload')

# Temporary File Handler

In [14]:
def get_uploaded_image_path(uploader_widget):
    if not uploader_widget.value or len(uploader_widget.value) == 0:
        raise ValueError("No ECG image uploaded")

    uploaded = uploader_widget.value[0]   # PyCharm tuple format

    temp_dir = tempfile.mkdtemp()
    temp_path = Path(temp_dir) / uploaded["name"]

    with open(temp_path, "wb") as f:
        f.write(uploaded["content"])

    return temp_path

# Process Image

In [15]:
img_path = get_uploaded_image_path(uploader)

vec = process_single_ecg_image(
    str(img_path),
    target_len=TARGET_LEAD_LENGTH
)

print("Vector shape:", vec.shape)

Vector shape: (8844,)


# Build DataFrame

In [16]:
columns = [
    f"Lead{lead}_{i}"
    for lead in range(1, 13)
    for i in range(TARGET_LEAD_LENGTH)
]

df = pd.DataFrame([vec], columns=columns)

# Save Dynamically

In [17]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

output_file = OUTPUT_DIR / f"{img_path.stem}_preprocessed.csv"

df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: outputs\Normal(1) - Copy_preprocessed.csv
